# 01 — Data Acquisition

This notebook inspects the bundled Delhi monitoring data and optionally refreshes it from XKDR.
Run the cells in order. The refresh cell is disabled by default because it accesses the network and regenerates the combined raw CSV.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import pandas as pd

stations = pd.read_csv(PROJECT_ROOT / "data/raw/selected_stations.csv")
metadata = pd.read_csv(PROJECT_ROOT / "data/raw/delhi_stations_metadata.csv")

print("Selected stations:", len(stations))
display(stations[["station_id", "name", "zone", "lat", "lon"]])
print("Downloaded station metadata:", len(metadata))


## Inspect the bundled observations

The 2023–2026 file is a combined raw dataset. Keep `delhi_pm25_2023_2026_raw.csv` only for the extended project workflow; the refresh script may synthesize missing hourly records to complete that timeline. Use the 2023–2024 raw file for source-observation-only work.


In [ ]:
from pathlib import Path

raw_dir = PROJECT_ROOT / "data/raw"
raw_files = sorted(raw_dir.glob("*.csv"))
for path in raw_files:
    frame = pd.read_csv(path)
    print(f"{path.name}: {len(frame):,} rows, {len(frame.columns)} columns")

observations = pd.read_csv(raw_dir / "delhi_pm25_2023_2026_raw.csv")
observations["timestamp"] = pd.to_datetime(observations["collected_at"])
print("Observation range:", observations["timestamp"].min(), "to", observations["timestamp"].max())
display(observations.head())


## Optional live refresh

Set `RUN_ACQUISITION = True` only when you intentionally want to query XKDR and rewrite the combined raw dataset. This step can take a long time.


In [ ]:
RUN_ACQUISITION = False

if RUN_ACQUISITION:
    import fetch_measurements
    fetch_measurements.main()
else:
    print("Skipped network refresh. Bundled raw data remains unchanged.")
